# LSTM Experimentation
- **Group Name**: MVPs
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
---

This notebook develops and evaluates the recurrent model required for midterm project. An LSTM trained from scratch classifies emails as spam or ham using the team's shared preprocessing and experimental framework.

Experiments include the main model, a controlled ablation, three training seeds, and inference-cost measurements. Model selection uses validation data; the test set is reserved for final evaluation.

## 1. Initial Configuration

### 1.1 Import, Paths and Device Configuration

In [15]:
# Import required libraries
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import torch
from torch import nn
from warnings import filterwarnings

# Ignore warnings
filterwarnings('ignore')

In [16]:
# Locate the repository from either its root or the notebooks folder to access local modules
current_dir = Path.cwd().resolve()

PROJECT_ROOT = (current_dir if (current_dir / 'src').is_dir() else current_dir.parent)

if not (PROJECT_ROOT / 'src').is_dir():raise FileNotFoundError('Open the notebook from the repository root or notebooks folder.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [17]:
# Import local modules
from src.reproducibility import set_seed
from src.data import load_enron_dataset;

In [18]:
# Set device dynamically to support CPU and GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Print environment details
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Python: 3.12.10
PyTorch: 2.9.1+cu128
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


### 1.2 Reproducibility

The data split uses a fixed seed so that every architecture is evaluated on the same examples. Separate training seeds are used to measure variation caused by initialization and training randomness.

Each experiment resets the random generators before creating the model and training data loader. Seeds improve repeatability, although identical results are not guaranteed across different hardware or software versions.

In [19]:
# Set seeds for reproducibility, constant across experiments
SPLIT_SEED = 42
TRAINING_SEEDS = [42, 123, 2026]

set_seed(TRAINING_SEEDS[0])

print(f'Split seed: {SPLIT_SEED}')
print(f'Training seeds: {TRAINING_SEEDS}')

Split seed: 42
Training seeds: [42, 123, 2026]


## 2. Data Loading and Preparation

### 2.1. Load the Source Dataset

The Enron spam dataset is loaded with its original partitions preserved. Its structure is inspected before defining the shared experimental splits. No vocabulary fitting or model training is performed at this stage.

In [20]:
# Define the dataset ID and revision
DATASET_ID = 'SetFit/enron_spam'
DATASET_REVISION = 'main'

In [22]:
# Load the dataset
raw_dataset = load_enron_dataset(cache_dir = PROJECT_ROOT / 'data' / 'hf_cache',  revision = DATASET_REVISION)
print(raw_dataset)

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 31716
    })
    test: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 2000
    })
})


In [ ]:
# Inspect the dataset structure for training and test partitions
for split_name, split_data in raw_dataset.items():
    print(f'Partition: {split_name}')
    print(f'Rows: {len(split_data):,}')
    print(f'Columns: {split_data.column_names}')
    print(f'Features: {split_data.features}\n')

Partition: train
Rows: 31,716
Columns: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']
Features: {'message_id': Value('int64'), 'text': Value('string'), 'label': Value('int64'), 'label_text': Value('string'), 'subject': Value('string'), 'message': Value('string'), 'date': Value('timestamp[s]')}

Partition: test
Rows: 2,000
Columns: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']
Features: {'message_id': Value('int64'), 'text': Value('string'), 'label': Value('int64'), 'label_text': Value('string'), 'subject': Value('string'), 'message': Value('string'), 'date': Value('timestamp[s]')}

